### 1.2 Формулировка

**Входы**

- Неформальное описание задания
- Результаты предварительного анализа и разметка полей

**Выходы**

- Формальная постановка задачи прогнозирования loan_status
- Сценарий использования, входные признаки и результат модели
- Требования к данным и моменту доступности признаков
- Критерий выбора модели, правило выбора порога и метрики итоговой оценки

**Что делаем**

По результатам изучения данных формулируем задачу, определяем входы и выход модели, сценарий её применения и критерий оценивания


In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


##### Сценарий использования модели


#### Обоснование постановки

Ввиду нескольких возможных интерпретаций задачи и датасета, остановимся на таком варианте:

Заёмщик делает заявку на кредит, предоставляя кредитору свои данные, желаемую сумму, цель займа и так далее

Кредитор выставляет потенциальному займу рейтинг (loan_grade) и на его основе предлагает процентную ставку (loan_int_rate)

Кредитор делает это с помощью собственного прогноза — своей моделью или вручную

При этом можно полагать, что кредитору доступно гораздо больше признаков, чем представлено в датасете нам, поэтому loan_grade (и связанный с ним loan_int_rate) — это выжимка из большего набора признаков, и мы её учитываем в нашей модели

loan_status, с другой стороны, — реальный факт, перестал ли заёмщик платить; наша модель будет предсказывать именно его

Зачем это нужно? — К примеру, если кредитор выступает посредником между заёмщиком и инвесторами, которые на основе оценки кредитора и ограниченных данных о займе и заёмщике сами решают, финансировать ли кредит

В этом случае инвестор нуждается в нашей модели, чтобы оценивать свои риски

Итак, задача — предсказывать значение loan_status на новых данных (все остальные столбцы — признаки)

Поскольку поле loan_status бинарное, это задача классификации, а конкретно бинарной классификации


#### Входы и результат модели

В разметке `loan_status` наблюдаются два значения: 0 — отсутствие дефолта, 1 — дефолт. Поэтому сформулированная задача относится к бинарной классификации; положительный класс — 1.

| Элемент постановки | Определение |
| --- | --- |
| Объект прогноза | Одна заявка на кредит |
| Вход модели | Все 11 столбцов, кроме `loan_status`: сведения о заёмщике, кредитной истории и условиях займа |
| Результат модели | Оценка риска дефолта; по выбранному порогу — класс 0 или 1 |
| Пользователь прогноза | Инвестор, рассматривающий финансирование кредита |
| Момент прогноза | После назначения кредитором грейда `loan_grade` и ставки `loan_int_rate`, до получения фактического исхода кредита |
| Назначение | Сопоставить заявки по риску неплатежа и учесть этот риск при выборе финансирования |

Входы и результат задают, **что модель принимает и что возвращает**. Сценарий применения объясняет, **кто использует прогноз, в какой момент и для какого решения**. Конкретный алгоритм обучения выбирается после анализа данных.

#### Требования к данным

- Для обучения нужны прошлые заявки с известным исходом `loan_status`. Этот столбец используется как целевая метка и исключается из входных признаков.
- При применении модели нужны те же 11 полей с тем же смыслом и единицами измерения, что в обучающем наборе. Например, процентная ставка и доля кредита от дохода представлены по-разному: `loan_int_rate` — в процентах, `loan_percent_income` — долей от 0 до 1.
- Значения признаков должны быть доступны **на момент прогноза**. В выбранном сценарии грейд и ставка уже назначены кредитором, а сведения о прошлых дефолтах относятся к предшествующей кредитной истории. Фактический исход рассматриваемого кредита на вход не подаётся.
- Пропуски, дубликаты, недопустимые значения и согласованность связанных полей проверяются при EDA. Правила обработки выбираются по результатам анализа; параметры преобразований оцениваются только на train.

Источник не уточняет период наблюдения, за который определяется дефолт. Модель прогнозирует метку в её исходном определении; фиксированный горизонт риска из этих данных установить нельзя.

#### Критерий оценивания

| Решение | Критерий |
| --- | --- |
| Сравнение моделей | Максимальная **Average Precision (AP)** на validation — качество ранжирования заявок по риску дефолта, не зависящее от порога классификации |
| Выбор порога для выбранной модели | Максимальная **F2** на validation, с большим весом полноты обнаружения дефолтов |
| Итоговая оценка | AP на test; дополнительно ROC-AUC, precision, recall, F1, F2, accuracy, balanced accuracy, Brier и матрица ошибок |

F2 выбрана, чтобы придать больший вес обнаружению дефолтов. Стоимости ложной тревоги и пропущенного дефолта не заданы, поэтому этот критерий не является оптимизацией финансовых потерь.

Задание не устанавливает минимально допустимое значение метрики. Качество сопоставляем с базовой моделью Dummy, использующей доли классов обучающей выборки. Модель и порог выбираем по validation, test оставляем для итоговой оценки зафиксированного решения.